# **Track Network - Mechanical Switch**

### Data Fetching

In [64]:
import pandas as pd
import psycopg2

def fetch_table_to_dataframe(host_ip, database_name, user, password, table_name, port=5432):
    """
    Connects to PostgreSQL and loads the given table into a Pandas DataFrame.
    """
    try:
        # Connect to PostgreSQL
        connection = psycopg2.connect(
            host=host_ip,
            database=database_name,
            user=user,
            password=password,
            port=port
        )
        print(f"Connected successfully to {database_name} on {host_ip}")

        # Create query
        query = f"SELECT * FROM {table_name};"

        # Load into pandas DataFrame
        df = pd.read_sql_query(query, connection)
        print(f"✅ Fetched {len(df)} rows from '{table_name}'")

        return df

    except Exception as e:
        print(f"❌ Error: {e}")
        return None

    finally:
        if connection:
            connection.close()

# --- Configuration (same as before) ---
HOST_IP = "100.95.110.69"
DATABASE_NAME = "pradigma-extractor"
USER = "postgres"
PASSWORD = "password"
PORT = 5432
TABLE_NAME = "extraction"

df_original = fetch_table_to_dataframe(HOST_IP, DATABASE_NAME, USER, PASSWORD, TABLE_NAME, PORT)

✅ Fetched 7760 rows from 'extraction'


In [65]:
keywords = ["MechanicalSwitch"]

pattern = '|'.join(keywords)

df = df_original.copy(deep=True)
df = df[
    (df['status_id'] == 1) &
    (df['dept_name'] == 'Track-Network') &
    (df['filename'].str.contains(pattern, case=False, na=False))
][['filename', 'workorder_id', 'json_data']]

df.head(5)


,filename,workorder_id,json_data
471,TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,4.000485e+09,"{'notification': {'notification_no': 'NA', 'no..."
638,TN_PM_MTH_MechanicalSwitch1_4000490559.pdf,4.000491e+09,"{'notification': {'notification_no': 'NA', 'no..."
807,TN_PM_MTH_MechanicalSwitch7_4000528340.pdf,4.000528e+09,"{'notification': {'notification_no': 'NA', 'no..."
816,TN_PM_MTH_MechanicalSwitch7_4000545727.pdf,4.000546e+09,"{'notification': {'notification_no': 'NA', 'no..."
821,TN_PM_MTH_MechanicalSwitch4_4000479274.pdf,4.000479e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [66]:
import re

number_group_map = {
    range(1, 5): 'mechanical_switch_hitachi',
    range(5, 7): 'mechanical_switch_mhedemag',  
    range(7, 8): 'mechanical_switch',  
    range(8, 9): 'mechanical_switch_srb',
}

def get_mech_target(filename):
    match = re.search(r"mechanicalswitch(\d)", filename, re.IGNORECASE)
    if not match:
        return None
    num = int(match.group(1))
    for key_range, target in number_group_map.items():
        if num in key_range:
            return target
    return None

def rename_json(row):
    data = row['json_data']
    if not isinstance(data, dict):
        return data
    
    target = get_mech_target(row['filename'])
    if target:
        return {target if k.startswith('mechanical_switch') else k: v for k, v in data.items()}

df['json_data'] = df.apply(rename_json, axis=1)

df


,filename,workorder_id,json_data
471,TN_PM_MTH_MechanicalSwitch1_4000484591.pdf,4.000485e+09,"{'notification': {'notification_no': 'NA', 'no..."
638,TN_PM_MTH_MechanicalSwitch1_4000490559.pdf,4.000491e+09,"{'notification': {'notification_no': 'NA', 'no..."
807,TN_PM_MTH_MechanicalSwitch7_4000528340.pdf,4.000528e+09,"{'notification': {'notification_no': 'NA', 'no..."
816,TN_PM_MTH_MechanicalSwitch7_4000545727.pdf,4.000546e+09,"{'notification': {'notification_no': 'NA', 'no..."
821,TN_PM_MTH_MechanicalSwitch4_4000479274.pdf,4.000479e+09,"{'notification': {'notification_no': 'NA', 'no..."
...,...,...,...
7478,TN_PM_MTH_MechanicalSwitch7_4000484597.pdf,4.000485e+09,"{'notification': {'notification_no': 'NA', 'no..."
7479,TN_PM_MTH_MechanicalSwitch7_4000501533.pdf,4.000502e+09,"{'notification': {'notification_no': 'NA', 'no..."
7480,TN_PM_MTH_MechanicalSwitch7_4000539710.pdf,4.000540e+09,"{'notification': {'notification_no': 'NA', 'no..."
7482,TN_PM_MTH_MechanicalSwitch7_4000523601.pdf,4.000524e+09,"{'notification': {'notification_no': 'NA', 'no..."


In [67]:
import pandas as pd

valid_json = df[df['json_data'].apply(lambda x: isinstance(x, dict))]

# initialize dictionary
key_to_workorders = {}

for _, row in valid_json.iterrows():
    workorder = row['workorder_id']
    data = row['json_data']
    
    for key in data.keys():
        key_to_workorders.setdefault(key, []).append(workorder)

for key, wos in key_to_workorders.items():
    unique_wos = sorted({int(x) for x in wos})
    # print(f"{key} ({len(unique_wos)}): {unique_wos}")
    print(f"{key} ({len(unique_wos)})")



notification (335)
work_order (335)
mechanical_switch_hitachi (162)
mechanical_switch (42)
mechanical_switch_srb (44)
mechanical_switch_mhedemag (87)


In [68]:
import os
import re
import json
import numpy as np
import pandas as pd
from collections import Counter

na_like_values = ['NA', 'N/A', 'NULL', 'NONE', 'NAN']
pattern_na = re.compile(r'^\s*(NA|N/A|NULL|NaN)\s*$', re.IGNORECASE)

def is_na_like(val):
    """Detect NA-like values."""
    if isinstance(val, (list, dict, np.ndarray)):
        return False
    try:
        if pd.isna(val):
            return True
    except Exception:
        pass
    val_str = str(val).strip().upper()
    return val_str in na_like_values

def find_na_keys(d):
    """Return keys in dict where value is NA-like."""
    if not isinstance(d, dict):
        return []
    return [k for k, v in d.items() if is_na_like(v)]


def clean_value(val):
    """Recursively clean NA-like values in dict, list, string."""
    if isinstance(val, str):
        return '' if pattern_na.match(val) else val
    elif isinstance(val, dict):
        return {k: clean_value(v) for k, v in val.items()}
    elif isinstance(val, list):
        return [clean_value(v) for v in val]
    else:
        return '' if pd.isna(val) else val

def extract_leaf_keys(d, parent=''):
    """Extract flattened leaf keys from nested dict."""
    keys = []
    if isinstance(d, dict):
        for k, v in d.items():
            full_key = f"{parent}.{k}" if parent else k
            if isinstance(v, dict):
                keys.extend(extract_leaf_keys(v, full_key))
            else:
                keys.append(full_key)
    return keys

def flatten_with_descriptions(subdict):
    """Flatten JSON dict, incorporating 'description' keys as part of flattened column names."""
    flat = {}

    def recurse(d, parent=''):
        if d is None:
            return
        if isinstance(d, str):
            try:
                d = json.loads(d)
            except json.JSONDecodeError:
                return
        if not isinstance(d, dict):
            return

        for k, v in d.items():
            if len(k) == 1 and k.isalpha():
                new_parent = parent
            else:
                new_parent = f"{parent}.{k}" if parent else k

            if isinstance(v, dict):
                desc = v.get('description')
                if desc:
                    desc_key = desc.lower().replace(' ', '_').replace('/', '_').replace('&', 'and')
                    for sub_k, sub_v in v.items():
                        if sub_k != 'description':
                            flat[f"{new_parent}.{desc_key}.{sub_k}"] = sub_v
                else:
                    recurse(v, new_parent)
            else:
                flat[new_parent] = v

    recurse(subdict)
    return flat

### Mechanical Switch (Hitachi)

In [69]:
import string

def flatten_json(data):
    flat = {}

    for section, items in data.items():
        if not isinstance(items, dict):
            flat[f"{section}"] = items
            continue

        labels = list(string.ascii_lowercase)

        for idx, (original_key, values) in enumerate(items.items()):
            label = labels[idx]
            base = f"{section}.{label}"

            if isinstance(values, dict):
                for k, v in values.items():
                    clean_k = (
                        "type" if k == "type_of_inspection"
                        else k
                    )
                    flat[f"{base}.{clean_k}"] = v
            else:
                flat[f"{base}"] = values

    return flat

df_mechswitch_hitachi = df.copy()

df_mechswitch_hitachi['mechanical_switch_hitachi'] = df_mechswitch_hitachi['json_data'].apply(
    lambda x: x.get('mechanical_switch_hitachi') if isinstance(x, dict) else None
)

df_mechswitch_hitachi = df_mechswitch_hitachi[df_mechswitch_hitachi['mechanical_switch_hitachi'].notnull()].copy()

df_mechswitch_hitachi['workorder_id'] = df_mechswitch_hitachi['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_mechswitch_hitachi['mechanical_switch_hitachi'] = df_mechswitch_hitachi['mechanical_switch_hitachi'].apply(clean_value)

df_mechswitch_hitachi['na_keys'] = df_mechswitch_hitachi['mechanical_switch_hitachi'].apply(find_na_keys)
na_counter = Counter(k for keys in df_mechswitch_hitachi['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flatten_json(r) for r in df_mechswitch_hitachi['mechanical_switch_hitachi'].fillna({})]
mechswitch_hitachi = pd.DataFrame(flattened_rows)  # final DataFrame
mechswitch_hitachi.index = df_mechswitch_hitachi.index
mechswitch_hitachi['workorder_id'] = df_mechswitch_hitachi['workorder_id'].astype('Int64')
mechswitch_hitachi['filename'] = df_mechswitch_hitachi['filename']

mechswitch_hitachi.columns = (
    mechswitch_hitachi.columns
    .str.replace(r'_cont(?=\.)', '', regex=True)
)

dup_cols = mechswitch_hitachi.columns[mechswitch_hitachi.columns.duplicated()]
rename_map = {
    col: col.replace(".a.", ".g.")
    for col in dup_cols
}

mechswitch_hitachi = mechswitch_hitachi.rename(columns=rename_map)
mechswitch_hitachi = mechswitch_hitachi.rename(columns={
        "technician.a": "technician_id",
        "technician.b": "technician_date",
        "supervisor.a": "supervisor_id",
        "supervisor.b": "supervisor_date",
    })

for i, col in enumerate(mechswitch_hitachi.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    
    col_data = mechswitch_hitachi.iloc[:, i-1]  # get column by index (always Series)
    valid_workorders = mechswitch_hitachi.loc[col_data.notna(), 'workorder_id'].unique()
    
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. non_rumba_mechanism.a.type
   Work Orders with data (162): 4000484591, 4000490559, 4000479274, 4000517926, 4000564479, 4000512640, 4000528334, 4000535226, 4000523595, 4000517925, 4000551629, 4000557883, 4000447706, 4000583474, 4000462344, 4000506768, 4000501526, 4000577229, 4000451932, 4000442319, 4000456676, 4000673670, 4000459932, 4000454478, 4000667956, 4000583473, 4000648842, 4000479272, 4000602420, 4000606465, 4000625910, 4000630379, 4000654660, 4000680435, 4000447708, 4000454760, 4000460253, 4000464888, 4000473449, 4000479273, 4000490581, 4000523597, 4000535228, 4000539655, 4000528336, 4000557885, 4000564480, 4000570745, 4000648844, 4000654661, 4000673672, 4000660922, 4000686326, 4000449819, 4000460287, 4000465095, 4000484594, 4000473450, 4000512642, 4000506771, 4000517928, 4000528337, 4000588672, 4000612359, 4000630382, 4000625942, 4000638199, 4000654663, 4000660923, 4000673673, 4000680439, 4000686327, 4000693791, 4000570741, 4000473447, 4000686324, 4000447707, 4000449617, 

In [70]:
# Rename all the columns based on description keys
checklist_items = {
    "non_rumba_mechanism.a.type": "non_rumba_mechanism.e.type",
    "non_rumba_mechanism.a.inspection_item": "non_rumba_mechanism.e.inspection_item",
    "non_rumba_mechanism.a.completed?": "non_rumba_mechanism.e.completed?",
    "non_rumba_mechanism.b.type": "non_rumba_mechanism.f.type",
    "non_rumba_mechanism.b.inspection_item": "non_rumba_mechanism.f.inspection_item",
    "non_rumba_mechanism.b.completed?": "non_rumba_mechanism.f.completed?",
    "non_rumba_mechanism.c.type": "non_rumba_mechanism.g.type",
    "non_rumba_mechanism.c.inspection_item": "non_rumba_mechanism.g.inspection_item",
    "non_rumba_mechanism.c.completed?": "non_rumba_mechanism.g.completed?",
}

mechswitch_hitachi = mechswitch_hitachi.rename(columns=checklist_items)

In [71]:
# create several new columns and pre-filled with default values
mechswitch_hitachi['non_rumba_mechanism.a.inspection_item'] = 'Non-Rumba motor and gearbox securely mounted.'
mechswitch_hitachi['non_rumba_mechanism.a.completed?'] = 'yes'
mechswitch_hitachi['non_rumba_mechanism.b.inspection_item'] = 'Frame is securely mounted to Switch beam underside.'
mechswitch_hitachi['non_rumba_mechanism.b.completed?'] = 'yes'
mechswitch_hitachi['non_rumba_mechanism.c.inspection_item'] = 'Limit switches secure and striker arms return.'
mechswitch_hitachi['non_rumba_mechanism.c.completed?'] = 'yes'
mechswitch_hitachi['non_rumba_mechanism.d.inspection_item'] = 'Non-Rumba arm secure and locking nuts tight.'
mechswitch_hitachi['non_rumba_mechanism.d.completed?'] = 'yes'

mechswitch_hitachi['switch_deck_and_rc_beam.a.inspection_item'] = 'Locking pin recesses drained of water.'
mechswitch_hitachi['switch_deck_and_rc_beam.a.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.b.inspection_item'] = 'Locking pin guide plate clearance 1mm maximum between cam roller and plates.'
mechswitch_hitachi['switch_deck_and_rc_beam.b.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.c.inspection_item'] = 'Carriage rail retaining bolts tight and clamping rail to plinth.'
mechswitch_hitachi['switch_deck_and_rc_beam.c.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.d.inspection_item'] = 'Switch deck drainage clean.'
mechswitch_hitachi['switch_deck_and_rc_beam.d.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.e.inspection_item'] = 'Switch deck auxiliaries fastened to the switch deck.'
mechswitch_hitachi['switch_deck_and_rc_beam.e.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.f.inspection_item'] = 'Finger plates bolted securely.'
mechswitch_hitachi['switch_deck_and_rc_beam.f.completed?'] = 'yes'
mechswitch_hitachi['switch_deck_and_rc_beam.g.inspection_item'] = 'Lighting is adequate.'
mechswitch_hitachi['switch_deck_and_rc_beam.g.completed?'] = 'yes'

mechswitch_hitachi['carriage_including_carriage_wheel.a.inspection_item'] = 'Carriage wheel rotates.'
mechswitch_hitachi['carriage_including_carriage_wheel.a.completed?'] = 'yes'
mechswitch_hitachi['carriage_including_carriage_wheel.b.inspection_item'] = 'Carriage wheel axle retainer.'
mechswitch_hitachi['carriage_including_carriage_wheel.b.completed?'] = 'yes'

mechswitch_hitachi['locking_pin_assembly.a.inspection_item'] = 'Locking pin motor secured to mounting frame.'
mechswitch_hitachi['locking_pin_assembly.a.completed?'] = 'yes'
mechswitch_hitachi['locking_pin_assembly.b.inspection_item'] = 'Locking pin frame securely bolted to the Switch beam assembly.'
mechswitch_hitachi['locking_pin_assembly.b.completed?'] = 'yes'
mechswitch_hitachi['locking_pin_assembly.c.inspection_item'] = 'Guard around locking pin assembly secure and not damaged.'
mechswitch_hitachi['locking_pin_assembly.c.completed?'] = 'yes'
mechswitch_hitachi['locking_pin_assembly.d.inspection_item'] = 'Locking pin cam roller free to turn and grease seal in place.'
mechswitch_hitachi['locking_pin_assembly.d.completed?'] = 'yes'
mechswitch_hitachi['locking_pin_assembly.e.inspection_item'] = 'Limit switches secure and striker arms return.'
mechswitch_hitachi['locking_pin_assembly.e.completed?'] = 'yes'

### Mechanical Switch (SRB)

In [72]:
import string

def flatten_json(data):
    flat = {}

    for section, items in data.items():
        if not isinstance(items, dict):
            flat[f"{section}"] = items
            continue

        labels = list(string.ascii_lowercase)

        for idx, (original_key, values) in enumerate(items.items()):
            label = labels[idx]
            base = f"{section}.{label}"

            if isinstance(values, dict):
                for k, v in values.items():
                    clean_k = (
                        "type" if k == "type_of_inspection"
                        else k
                    )
                    flat[f"{base}.{clean_k}"] = v
            else:
                flat[f"{base}"] = values

    return flat

df_mechswitch_srb = df.copy()

df_mechswitch_srb['mechanical_switch_srb'] = df_mechswitch_srb['json_data'].apply(
    lambda x: x.get('mechanical_switch_srb') if isinstance(x, dict) else None
)

df_mechswitch_srb = df_mechswitch_srb[df_mechswitch_srb['mechanical_switch_srb'].notnull()].copy()

df_mechswitch_srb['workorder_id'] = df_mechswitch_srb['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_mechswitch_srb['mechanical_switch_srb'] = df_mechswitch_srb['mechanical_switch_srb'].apply(clean_value)

df_mechswitch_srb['na_keys'] = df_mechswitch_srb['mechanical_switch_srb'].apply(find_na_keys)
na_counter = Counter(k for keys in df_mechswitch_srb['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flatten_json(r) for r in df_mechswitch_srb['mechanical_switch_srb'].fillna({})]
mechswitch_srb = pd.DataFrame(flattened_rows)
mechswitch_srb.index = df_mechswitch_srb.index
mechswitch_srb['workorder_id'] = df_mechswitch_srb['workorder_id'].astype('Int64')
mechswitch_srb['filename'] = df_mechswitch_srb['filename']

mechswitch_srb = mechswitch_srb.rename(columns={
        "technician.a": "technician_id",
        "technician.b": "technician_date",
        "supervisor.a": "supervisor_id",
        "supervisor.b": "supervisor_date",
    })

for i, col in enumerate(mechswitch_srb.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = mechswitch_srb.loc[mechswitch_srb[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. local_control_cabinet.a.type
   Work Orders with data (44): 4000453205, 4000643680, 4000506777, 4000442337, 4000447697, 4000462639, 4000467953, 4000473454, 4000490586, 4000479281, 4000495733, 4000517932, 4000523602, 4000528341, 4000535233, 4000545728, 4000551638, 4000570806, 4000564486, 4000577264, 4000588680, 4000597618, 4000602910, 4000612365, 4000619035, 4000625951, 4000638206, 4000648850, 4000654668, 4000667967, 4000673677, 4000680443, 4000686331, 4000693795, 4000457977, 4000512646, 4000484598, 4000501534, 4000539711, 4000557890, 4000583480, 4000606476, 4000630386, 4000660929
--------------------------------------------------------------------------------
  2. local_control_cabinet.a.description
   Work Orders with data (44): 4000453205, 4000643680, 4000506777, 4000442337, 4000447697, 4000462639, 4000467953, 4000473454, 4000490586, 4000479281, 4000495733, 4000517932, 4000523602, 4000528341, 4000535233, 4000545728, 4000551638, 4000570806, 4000564486, 4000577264, 4000588680, 400

### Mechanical Switch (MHE-DEMAG)

In [73]:
import string

def flatten_json(data):
    """
    Flatten JSON for mechanical switch SRB with clean column names:
        section_a_type
        section_a_description
        section_a_completed
    """
    flat = {}

    for section, items in data.items():
        if not isinstance(items, dict):
            flat[f"{section}"] = items
            continue

        labels = list(string.ascii_lowercase)

        for idx, (original_key, values) in enumerate(items.items()):
            label = labels[idx]
            base = f"{section}.{label}"

            if isinstance(values, dict):
                for k, v in values.items():
                    clean_k = (
                        "type" if k == "type_of_inspection"
                        else k
                    )
                    flat[f"{base}.{clean_k}"] = v
            else:
                flat[f"{base}"] = values

    return flat

df_mechswitch_mhedemag = df.copy()

df_mechswitch_mhedemag['mechanical_switch_mhedemag'] = df_mechswitch_mhedemag['json_data'].apply(
    lambda x: x.get('mechanical_switch_mhedemag') if isinstance(x, dict) else None
)

df_mechswitch_mhedemag = df_mechswitch_mhedemag[df_mechswitch_mhedemag['mechanical_switch_mhedemag'].notnull()].copy()

df_mechswitch_mhedemag['workorder_id'] = df_mechswitch_mhedemag['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_mechswitch_mhedemag['mechanical_switch_mhedemag'] = df_mechswitch_mhedemag['mechanical_switch_mhedemag'].apply(clean_value)

df_mechswitch_mhedemag['na_keys'] = df_mechswitch_mhedemag['mechanical_switch_mhedemag'].apply(find_na_keys)
na_counter = Counter(k for keys in df_mechswitch_mhedemag['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flatten_json(r) for r in df_mechswitch_mhedemag['mechanical_switch_mhedemag'].fillna({})]
mechswitch_mhedemag = pd.DataFrame(flattened_rows)
mechswitch_mhedemag.index = df_mechswitch_mhedemag.index
mechswitch_mhedemag['workorder_id'] = df_mechswitch_mhedemag['workorder_id'].astype('Int64')
mechswitch_mhedemag['filename'] = df_mechswitch_mhedemag['filename']

mechswitch_mhedemag = mechswitch_mhedemag.rename(columns={
        "technician.a": "technician_id",
        "technician.b": "technician_date",
        "supervisor.a": "supervisor_id",
        "supervisor.b": "supervisor_date",
    })

for i, col in enumerate(mechswitch_mhedemag.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = mechswitch_mhedemag.loc[mechswitch_mhedemag[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. switch_deck_and_RC_beam.a.type
   Work Orders with data (87): 4000512644, 4000506774, 4000517930, 4000523600, 4000528339, 4000535231, 4000539660, 4000545724, 4000557888, 4000602908, 4000606474, 4000444720, 4000449820, 4000460288, 4000479275, 4000490583, 4000506772, 4000512643, 4000517929, 4000523599, 4000539657, 4000545723, 4000551634, 4000564482, 4000570802, 4000577238, 4000583477, 4000588675, 4000625947, 4000597615, 4000630383, 4000648846, 4000654665, 4000660924, 4000667962, 4000673674, 4000680440, 4000693792, 4000446596, 4000452677, 4000462637, 4000473452, 4000479277, 4000484596, 4000495731, 4000501532, 4000625949, 4000643675, 4000648848, 4000654666, 4000660926, 4000667964, 4000465096, 4000473451, 4000495730, 4000619032, 4000673675, 4000680441, 4000643673, 4000693793, 4000597616, 4000588676, 4000686329, 4000442122, 4000455291, 4000501531, 4000535230, 4000528338, 4000557887, 4000602907, 4000606473, 4000612361, 4000638200, 4000442309, 4000686328, 4000465154, 4000457722, 400049058

### Mechanical Switch 

In [74]:
import string

def flatten_json(data):
    """
    Flatten JSON for mechanical switch SRB with clean column names:
        section_a_type
        section_a_description
        section_a_completed
    """
    flat = {}

    for section, items in data.items():
        if not isinstance(items, dict):
            flat[f"{section}"] = items
            continue

        labels = list(string.ascii_lowercase)

        for idx, (original_key, values) in enumerate(items.items()):
            label = labels[idx]
            base = f"{section}.{label}"

            if isinstance(values, dict):
                for k, v in values.items():
                    clean_k = (
                        "type" if k == "type_of_inspection"
                        else k
                    )
                    flat[f"{base}.{clean_k}"] = v
            else:
                flat[f"{base}"] = values

    return flat

df_mechswitch = df.copy()

df_mechswitch['mechanical_switch'] = df_mechswitch['json_data'].apply(
    lambda x: x.get('mechanical_switch') if isinstance(x, dict) else None
)

df_mechswitch = df_mechswitch[df_mechswitch['mechanical_switch'].notnull()].copy()

df_mechswitch['workorder_id'] = df_mechswitch['workorder_id'].apply(lambda x: int(x) if pd.notnull(x) else None)

df_mechswitch['mechanical_switch'] = df_mechswitch['mechanical_switch'].apply(clean_value)

df_mechswitch['na_keys'] = df_mechswitch['mechanical_switch'].apply(find_na_keys)
na_counter = Counter(k for keys in df_mechswitch['na_keys'] for k in keys)
na_summary = pd.DataFrame(na_counter.items(), columns=['key', 'na_count']).sort_values('na_count', ascending=False)

flattened_rows = [flatten_json(r) for r in df_mechswitch['mechanical_switch'].fillna({})]
mechswitch = pd.DataFrame(flattened_rows)  # final DataFrame
mechswitch.index = df_mechswitch.index
mechswitch['workorder_id'] = df_mechswitch['workorder_id'].astype('Int64')
mechswitch['filename'] = df_mechswitch['filename']

mechswitch = mechswitch.rename(columns={
        "technician.a": "technician_id",
        "technician.b": "technician_date",
        "supervisor.a": "supervisor_id",
        "supervisor.b": "supervisor_date",
    })

for i, col in enumerate(mechswitch.columns, start=1):
    print(f"{i:3d}. {col}")
    if col in ['workorder_id', 'filename']:
        continue
    valid_workorders = mechswitch.loc[mechswitch[col].notna(), 'workorder_id'].unique()
    if len(valid_workorders) > 0:
        workorder_list = ", ".join(map(str, valid_workorders))
        print(f"   Work Orders with data ({len(valid_workorders)}): {workorder_list}")
        print("-" * 80)


  1. local_control_cabinet.a.type
   Work Orders with data (42): 4000528340, 4000545727, 4000648849, 4000517931, 4000457975, 4000462638, 4000495732, 4000512645, 4000564485, 4000570805, 4000557889, 4000583479, 4000597617, 4000588679, 4000606475, 4000612364, 4000619034, 4000630385, 4000654667, 4000638204, 4000673676, 4000660927, 4000680442, 4000693794, 4000551637, 4000577263, 4000602909, 4000625950, 4000667965, 4000686330, 4000447710, 4000442335, 4000453204, 4000473453, 4000467952, 4000479279, 4000506775, 4000484597, 4000501533, 4000539710, 4000523601, 4000535232
--------------------------------------------------------------------------------
  2. local_control_cabinet.a.description
   Work Orders with data (42): 4000528340, 4000545727, 4000648849, 4000517931, 4000457975, 4000462638, 4000495732, 4000512645, 4000564485, 4000570805, 4000557889, 4000583479, 4000597617, 4000588679, 4000606475, 4000612364, 4000619034, 4000630385, 4000654667, 4000638204, 4000673676, 4000660927, 4000680442, 400

### Output Excel

In [75]:
import os
import pandas as pd
from openpyxl import Workbook

output_path = '../../output/tnm/mechanical_switch_final.xlsx'

os.makedirs(os.path.dirname(output_path), exist_ok=True)

if not os.path.exists(output_path):
    Workbook().save(output_path)

with pd.ExcelWriter(output_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    mechswitch_hitachi.to_excel(writer, index=False, sheet_name='hitachi'),
    mechswitch_srb.to_excel(writer, index=False, sheet_name='srb'),
    mechswitch_mhedemag.to_excel(writer, index=False, sheet_name='mhe-demag'),
    mechswitch.to_excel(writer, index=False, sheet_name='mechanical-switch'),

print(f"✅ Exported successfully to '{output_path}' (replaced existing sheet)")


✅ Exported successfully to '../../output/tnm/mechanical_switch_final.xlsx' (replaced existing sheet)
